# Advanced Retrieval Methodologies

## Better retrieval before a bigger model

This notebook improves one retrieval pipeline layer by layer:

- structure-aware chunking instead of blind slicing
- source and section context on every chunk
- small child chunks for search and larger parent sections for answers
- vector search plus BM25 keyword search
- Reciprocal Rank Fusion and second-stage reranking
- query expansion and HyDE for vague questions

The objective is to understand **why each layer exists**. The implementation stays small enough to inspect in one notebook.

In [1]:
# Run once. The reranker model downloads the first time it is used.
%pip install -q openai pypdf rank-bm25 sentence-transformers python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, warnings
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

warnings.filterwarnings('ignore', message='.*flash attention.*')

ROOT = Path.cwd()
load_dotenv(ROOT / '.env')
load_dotenv(ROOT.parent / '.env')

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key: ')
CHAT_MODEL = os.getenv('OPENAI_CHAT_MODEL', 'gpt-5-mini')
EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
DATA_FOLDER = ROOT / 'data'

client = OpenAI(api_key=OPENAI_API_KEY)
print(f'Ready: {len(list(DATA_FOLDER.glob("*.pdf")))} PDFs | {CHAT_MODEL} | {EMBEDDING_MODEL}')

Ready: 3 PDFs | gpt-5-mini | text-embedding-3-small


## 1. Parse the document's structure

A fixed-size splitter knows only character or token counts. Our PDFs contain visible headings such as `SECTION 1 - NX-417 SENSOR LINK LOSS`. We will use those headings as parent boundaries.

Each parent section keeps its document name, page number, and heading. Smaller child passages remain linked to that parent.

In [3]:
def parse_pdf_sections(pdf_path):
    reader = PdfReader(pdf_path)
    sections = []

    for page_number, page in enumerate(reader.pages, start=1):
        lines = [line.strip() for line in (page.extract_text() or '').splitlines() if line.strip()]
        heading_index = next((i for i, line in enumerate(lines) if line.startswith('SECTION ')), None)
        if heading_index is None:
            continue  # Cover page

        heading = lines[heading_index]
        body = ' '.join(lines[heading_index + 1:])
        sections.append({
            'parent_id': f'{pdf_path.stem}-p{page_number}',
            'source': pdf_path.name,
            'page': page_number,
            'heading': heading,
            'text': body,
        })
    return sections

parents = []
for pdf_path in sorted(DATA_FOLDER.glob('*.pdf')):
    parents.extend(parse_pdf_sections(pdf_path))

print(f'Created {len(parents)} parent sections from {len(list(DATA_FOLDER.glob("*.pdf")))} PDFs')
pd.DataFrame(parents)[['source', 'page', 'heading']]

Created 9 parent sections from 3 PDFs


,source,page,heading
0,01_product_support_manual.pdf,2,SECTION 1 - NX-417 SENSOR LINK LOSS
1,01_product_support_manual.pdf,3,SECTION 2 - NX-471 MOTOR FEEDBACK MISMATCH
2,01_product_support_manual.pdf,4,SECTION 3 - DISPLAY AND POWER BEHAVIOR
3,02_firmware_release_notes.pdf,2,SECTION 1 - VERSION 4.8.0 KNOWN ISSUE UI-204
4,02_firmware_release_notes.pdf,3,SECTION 2 - VERSION 4.8.2 DISPLAY FIX
5,02_firmware_release_notes.pdf,4,SECTION 3 - VERSION 4.8.3 SENSOR TELEMETRY
6,03_field_service_handbook.pdf,2,SECTION 1 - SAFE RECOVERY FOR SENSOR FAULTS
7,03_field_service_handbook.pdf,3,SECTION 2 - ESCALATION AND EVIDENCE PACKAGE
8,03_field_service_handbook.pdf,4,SECTION 3 - OFFLINE SITE OPERATION


## 2. Enrich every child with context

A sentence such as `Wait 90 seconds before reconnecting the harness` is dangerous by itself. Which harness? For which fault?

We search small passages, but stamp each one with its document and parent heading. This improves retrieval without changing the original text.

In [4]:
def split_children(parent, words_per_child=70, overlap=15):
    words = parent['text'].split()
    children, start = [], 0
    while start < len(words):
        end = min(start + words_per_child, len(words))
        passage = ' '.join(words[start:end])
        child_number = len(children) + 1
        children.append({
            **{key: parent[key] for key in ['parent_id', 'source', 'page', 'heading']},
            'child_id': f"{parent['parent_id']}-c{child_number}",
            'text': passage,
            'search_text': f"Document: {parent['source']}\nSection: {parent['heading']}\n{passage}",
        })
        if end == len(words):
            break
        start = end - overlap
    return children

children = [child for parent in parents for child in split_children(parent)]

raw_document = '\n'.join(page.extract_text() or '' for page in PdfReader(DATA_FOLDER / '01_product_support_manual.pdf').pages)
fixed = [raw_document[i:i + 500] for i in range(0, len(raw_document), 500)]

print('A blind 500-character boundary:')
print(fixed[0][-120:].replace('\n', ' ') + '  || CUT ||  ' + fixed[1][:120].replace('\n', ' '))
print(f'\nStructure-aware result: {len(parents)} parents -> {len(children)} searchable children')
print('\nExample enriched child:\n')
print(next(item['search_text'] for item in children if 'Wait 90 seconds' in item['text']))

A blind 500-character boundary:
gies lesson. It contains overlapping concepts, exact identifiers, and detailed procedures so different retrieval methods  || CUT ||   can be compared.  Northstar Edge Controller - Product Support Manual Page 2 SECTION 1 - NX-417 SENSOR LINK LOSS NX-417 

Structure-aware result: 9 parents -> 28 searchable children

Example enriched child:

Document: 01_product_support_manual.pdf
Section: SECTION 1 - NX-417 SENSOR LINK LOSS
which indicates a motor feedback disagreement after motion begins. Common causes include a loose M12 sensor connector, moisture inside the harness, or a sensor that was reconnected before stored charge left the input circuit. Recovery procedure Stop the line and apply the approved lockout procedure. Inspect the sensor body and cable before touching the connector. Disconnect the primary position sensor at port J7. Wait 90 seconds before reconnecting the sensor


## 3. Hierarchical retrieval: search small, read big

Small child chunks are precise search targets. Once a child matches, we replace it with the full parent section before answering.

This separates **what we search** from **what the model reads**.

In [5]:
parents_by_id = {item['parent_id']: item for item in parents}
example_child = next(item for item in children if 'Wait 90 seconds' in item['text'])
example_parent = parents_by_id[example_child['parent_id']]

print('SEARCH TARGET - child passage')
print(example_child['text'])
print('\nANSWER CONTEXT - full parent section')
print(example_parent['heading'])
print(example_parent['text'])

SEARCH TARGET - child passage
which indicates a motor feedback disagreement after motion begins. Common causes include a loose M12 sensor connector, moisture inside the harness, or a sensor that was reconnected before stored charge left the input circuit. Recovery procedure Stop the line and apply the approved lockout procedure. Inspect the sensor body and cable before touching the connector. Disconnect the primary position sensor at port J7. Wait 90 seconds before reconnecting the sensor

ANSWER CONTEXT - full parent section
SECTION 1 - NX-417 SENSOR LINK LOSS
NX-417 means the controller stopped receiving a valid heartbeat from the primary position sensor for more than 1.5 seconds. Recognizing the fault The operator panel shows NX-417 and places the affected axis in a safe hold. The event log may also record intermittent sensor voltage or a missing heartbeat. Do not confuse NX-417 with NX-471, which indicates a motor feedback disagreement after motion begins. Common causes include a l

## 4. Hybrid search: meaning plus exact words

Dense vectors find concepts and paraphrases. BM25 keyword search is strong on exact identifiers such as `NX-417`, `UI-204`, and `J7`.

Reciprocal Rank Fusion combines the two rankings without trying to compare their incompatible raw scores.

In [6]:
def embed(texts):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    matrix = np.array([item.embedding for item in response.data])
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)

STOPWORDS = {'a', 'an', 'and', 'are', 'do', 'does', 'for', 'how', 'i', 'in', 'is', 'it', 'my', 'of', 'on', 'the', 'to', 'what'}

def tokenize(text):
    tokens = re.findall(r'[a-z0-9]+(?:-[a-z0-9]+)*', text.lower())
    return [token for token in tokens if token not in STOPWORDS]

child_vectors = embed([item['search_text'] for item in children])
bm25 = BM25Okapi([tokenize(item['search_text']) for item in children])

def ranked_indices(scores, limit):
    return list(np.argsort(scores)[::-1][:limit])

def retrieve_lists(query, limit=8):
    query_vector = embed([query])[0]
    dense_scores = child_vectors @ query_vector
    keyword_scores = bm25.get_scores(tokenize(query))
    return ranked_indices(dense_scores, limit), ranked_indices(keyword_scores, limit)

def fuse_rankings(*rankings, rrf_k=60, limit=8):
    fused = {}
    for ranking in rankings:
        for rank, index in enumerate(ranking, start=1):
            fused[index] = fused.get(index, 0) + 1 / (rrf_k + rank)
    ordered = sorted(fused, key=fused.get, reverse=True)[:limit]
    return [{**children[index], 'rrf_score': fused[index]} for index in ordered]

print(f'Indexed {len(children)} children with vectors and BM25')

Indexed 28 children with vectors and BM25


In [7]:
query = 'NX-417'  # Users often paste only an exact error code.
dense_order, keyword_order = retrieve_lists(query)
hybrid_results = fuse_rankings(dense_order, keyword_order)

rows = []
for method, order in [('VECTOR', dense_order), ('BM25', keyword_order)]:
    for rank, index in enumerate(order[:3], start=1):
        rows.append({'method': method, 'rank': rank, 'section': children[index]['heading']})
for rank, item in enumerate(hybrid_results[:3], start=1):
    rows.append({'method': 'HYBRID RRF', 'rank': rank, 'section': item['heading']})

pd.DataFrame(rows)

,method,rank,section
0,VECTOR,1,SECTION 1 - NX-417 SENSOR LINK LOSS
1,VECTOR,2,SECTION 2 - NX-471 MOTOR FEEDBACK MISMATCH
2,VECTOR,3,SECTION 1 - NX-417 SENSOR LINK LOSS
3,BM25,1,SECTION 1 - NX-417 SENSOR LINK LOSS
4,BM25,2,SECTION 1 - NX-417 SENSOR LINK LOSS
5,BM25,3,SECTION 1 - NX-417 SENSOR LINK LOSS
6,HYBRID RRF,1,SECTION 1 - NX-417 SENSOR LINK LOSS
7,HYBRID RRF,2,SECTION 1 - NX-417 SENSOR LINK LOSS
8,HYBRID RRF,3,SECTION 2 - NX-471 MOTOR FEEDBACK MISMATCH


## 5. Two-stage reranking

The first stage searches quickly and keeps a wider candidate set. A cross-encoder then reads each query and candidate together to make a more careful relevance judgment.

Production alternatives include hosted reranking APIs and larger domain-specific rerankers. We use a small local model so no additional API key is required.

In [8]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, limit=3):
    pairs = [(query, item['search_text']) for item in candidates]
    scores = reranker.predict(pairs)
    reranked = [{**item, 'rerank_score': float(score)} for item, score in zip(candidates, scores)]
    return sorted(reranked, key=lambda item: item['rerank_score'], reverse=True)[:limit]

def swap_children_for_parents(ranked_children, limit=2):
    selected, seen = [], set()
    for child in ranked_children:
        if child['parent_id'] not in seen:
            selected.append(parents_by_id[child['parent_id']])
            seen.add(child['parent_id'])
        if len(selected) == limit:
            break
    return selected

rerank_query = 'How do I safely recover from NX-417?'
rerank_dense, rerank_keyword = retrieve_lists(rerank_query)
rerank_candidates = fuse_rankings(rerank_dense, rerank_keyword)
reranked = rerank(rerank_query, rerank_candidates)
selected_parents = swap_children_for_parents(reranked)

print('Precise children after reranking:')
display(pd.DataFrame([{'score': round(x['rerank_score'], 3), 'section': x['heading']} for x in reranked]))
print('Full parents supplied for answering:')
display(pd.DataFrame([{'section': x['heading'], 'words': len(x['text'].split())} for x in selected_parents]))

Precise children after reranking:


,score,section
0,8.058,SECTION 1 - SAFE RECOVERY FOR SENSOR FAULTS
1,6.473,SECTION 1 - SAFE RECOVERY FOR SENSOR FAULTS
2,5.555,SECTION 3 - VERSION 4.8.3 SENSOR TELEMETRY


Full parents supplied for answering:


,section,words
0,SECTION 1 - SAFE RECOVERY FOR SENSOR FAULTS,181
1,SECTION 3 - VERSION 4.8.3 SENSOR TELEMETRY,120


## 6. Query transformation

Users rarely speak like documentation. Query expansion creates several explicit searches from one vague question. HyDE creates a short hypothetical answer whose embedding may sit closer to the real document language.

These are options, not mandatory steps. Use them when user wording and document wording regularly differ.

In [ ]:
transformation_cache = {}

def transform_question(question):
    if question in transformation_cache:
        return transformation_cache[question]

    expansion = client.responses.create(
        model=CHAT_MODEL,
        instructions='Write exactly three concise search queries, one per line. No numbering or explanation.',
        input=(
            'Collection: Northstar Edge Controller manuals, firmware notes, and field procedures. '
            f'Turn this vague support question into specific searches for that collection: {question}'
        ),
    )
    queries = [re.sub(r'^[-\d.)\s]+', '', line).strip() for line in expansion.output_text.splitlines() if line.strip()][:3]

    hyde_response = client.responses.create(
        model=CHAT_MODEL,
        instructions=(
            'Write one 60-90 word hypothetical Northstar Edge Controller manual passage that would answer the question. '
            'This is industrial controller documentation, not laptop, phone, or desktop support. '
            'Use industrial firmware and operator-display language. Do not use bullets, invent error codes, or claim the passage is factual.'
        ),
        input=question,
    )
    result = {'queries': queries, 'hyde': hyde_response.output_text.strip()}
    transformation_cache[question] = result
    return result

vague_question = 'Why does my screen keep going dark after the update?'
transformed = transform_question(vague_question)
print('Expanded searches:')
for item in transformed['queries']:
    print('-', item)
print()
print('HyDE passage:')
print(transformed['hyde'])

## 7. The complete retrieval pipeline

The final function uses every layer:

`transform question -> vector + BM25 -> RRF -> rerank -> parent context -> grounded answer`

For a larger application, each layer could be replaced independently without redesigning the entire system.

In [ ]:
def multi_query_candidates(question, limit=10):
    transformed = transform_question(question)
    search_queries = [question] + transformed['queries']
    vectors = embed(search_queries + [transformed['hyde']])
    combined = {}

    for query_text, query_vector in zip(search_queries, vectors[:-1]):
        dense = ranked_indices(child_vectors @ query_vector, limit)
        keyword = ranked_indices(bm25.get_scores(tokenize(query_text)), limit)
        for ranking in (dense, keyword):
            for rank, index in enumerate(ranking, start=1):
                combined[index] = combined.get(index, 0) + 1 / (60 + rank)

    hyde_dense = ranked_indices(child_vectors @ vectors[-1], limit)
    for rank, index in enumerate(hyde_dense, start=1):
        combined[index] = combined.get(index, 0) + 1 / (60 + rank)

    ordered = sorted(combined, key=combined.get, reverse=True)[:limit]
    return [{**children[index], 'rrf_score': combined[index]} for index in ordered]

def advanced_rag(question):
    candidates = multi_query_candidates(question)
    best_children = rerank(question, candidates, limit=4)
    best_parents = swap_children_for_parents(best_children, limit=2)

    context = '\n\n'.join(
        f"[{item['source']} > {item['heading']}, page {item['page']}]\n{item['text']}"
        for item in best_parents
    )
    response = client.responses.create(
        model=CHAT_MODEL,
        instructions='Answer only from the supplied context. Keep it under 130 words and cite source, section, and page.',
        input=f'Question: {question}\n\nContext:\n{context}',
    )

    print('Selected parent sections:')
    for item in best_parents:
        print(f"- {item['source']} > {item['heading']} (page {item['page']})")
    print()
    print('Answer:')
    print(response.output_text)
    return response.output_text

advanced_rag(vague_question)

## Takeaway

| Layer | Problem it solves |
|---|---|
| Structural chunking | Broken sections and procedures |
| Context enrichment | Orphan passages |
| Parent-child retrieval | Precision versus sufficient context |
| Hybrid search | Meaning versus exact identifiers |
| Reranking | Weak ordering in the first candidate set |
| Query expansion and HyDE | Vague user language |

A production pipeline does not need every layer for every question. Add a layer when evaluation shows a specific retrieval failure.